## Content:
1. Download attacked NR quality metric model (PaQ-2-PiQ)
2. IOI attack code
3. Attack examples
4. Evaluating on the NIPS 2017 dataset

In [1]:
import torch
from torchvision import transforms
from PIL import Image
import time
import torch
import numpy as np
import time
import cv2
import imageio
from tqdm import tqdm
from scipy import ndimage
import matplotlib.pyplot as plt
from torch.autograd import Variable
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
from torch.autograd import Variable
import os
import subprocess
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision
from torch.utils.data import DataLoader
from torch.autograd import Variable

# Monkey-patch torch.load to always use weights_only=False
original_load = torch.load
def patched_load(f, *args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(f, *args, **kwargs)

torch.load = patched_load
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Download attacked NR quality metric model (PaQ-2-PiQ)

In [3]:
# clean the dir if any
!rm ./img/Picture1.png
!rm ./img/.DS_STORE
!rm ./img/.ipynb_checkpoints

# download the model
!wget -O RoIPoolModel.pth -N https://github.com/baidut/PaQ-2-PiQ/releases/download/v1.0/RoIPoolModel-fit.10.bs.120.pth

# download a test image
!wget -N https://github.com/baidut/PaQ-2-PiQ/releases/download/v1.0/Picture1.jpg

# download the standalone version of code
!wget -N https://raw.githubusercontent.com/baidut/PaQ-2-PiQ_GAE/master/paq2piq_standalone.py

rm: ./img/Picture1.png: No such file or directory
rm: ./img/.DS_STORE: No such file or directory
for details.

--2026-05-28 18:41:59--  https://github.com/baidut/PaQ-2-PiQ/releases/download/v1.0/RoIPoolModel-fit.10.bs.120.pth
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/237024974/a1c42500-4755-11ea-9c0e-7bf2246fe9e5?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-28T23%3A16%3A54Z&rscd=attachment%3B+filename%3DRoIPoolModel-fit.10.bs.120.pth&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-28T22%3A16%3A29Z&ske=2026-05-28T23%3A16%3A54Z&sks=b&skv=2018-11-09&sig=9tsokX6Eu3tTUos6lBeHEFUDgMk6N3cbKpuayJaPaOI%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2

In [4]:
import cv2
import matplotlib.pyplot as plt
import torch
from paq2piq_standalone import *
from torch.autograd import Variable
import imageio
import os
import subprocess

cpu


/Users/yuchen/Desktop/repos/ioi-attack/ioi-attack/paq2piq_standalone.py:46: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  if backbone is 'resnet18':


In [5]:
IMAGE_NET_MEAN = [0.485, 0.456, 0.406]
IMAGE_NET_STD = [0.229, 0.224, 0.225]


class Transform:
    def __init__(self):
        # normalize = transforms.Normalize(mean=IMAGE_NET_MEAN, std=IMAGE_NET_STD)

        self._train_transform = transforms.Compose(
            [
                transforms.ToTensor(),
            ]
        )

        self._val_transform = transforms.Compose([transforms.ToTensor()])

    @property
    def train_transform(self):
        return self._train_transform

    @property
    def val_transform(self):
        return self._val_transform

In [18]:
model_state = torch.load('RoIPoolModel.pth', map_location=lambda storage, loc: storage, weights_only=False)
model = RoIPoolModel()
model.load_state_dict(model_state["model"])
model = model.to(device)
transform = Transform().val_transform
model.eval()
torch.set_grad_enabled(False) 

/Users/yuchen/.virtualenvs/nb/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/yuchen/.virtualenvs/nb/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [7]:
overall_model = InferenceModel(RoIPoolModel(), 'RoIPoolModel.pth')
overall_model.blk_size = (3, 5)

## IOI attack code

In [8]:
from scipy.signal import convolve2d

In [9]:
def std_convoluted(img, win_size):
    img = np.moveaxis(img, -1, 0)  # HWC -> CHW
    img2 = img**2
    kernel = np.ones(win_size)
    kernel = kernel / kernel.size

    conv = lambda x: convolve2d(x, kernel, mode="valid")

    img_mean = np.stack([conv(band) for band in img], axis=-1)
    img2_mean = np.stack([conv(band) for band in img2], axis=-1)

    img_mean[img_mean == 0] = 1

    return np.sqrt(np.clip((img2_mean - img_mean**2), 0, None)) / img_mean

In [10]:
def attack(im, lrlr):
  keep = 0.07
  f = std_convoluted(im.astype('float32'), (3, 3)).astype('float32')
  p1d = (1, 1, 1, 1)
  f = transforms.ToTensor()(f)
  f = f.unsqueeze_(0)
  f = F.pad(f, p1d, "constant", 0)
  f = f.squeeze().data.cpu().numpy().transpose(1, 2, 0)
  #f[f<0.0001] = 0
  f = (f / f.max())
  f[f<0.01] = 0.0
  f = f ** 0.5
  #print(im)
  Rt = np.fft.fft2(im[:,:,0])
  Rtsort = np.sort(np.abs(Rt.reshape(-1)))
  tresh = Rtsort[int(np.floor((1-keep)*len(Rtsort)))]
  ind = np.abs(Rt)>tresh
  indr = np.abs(Rt)<tresh
  Rtlow = Rt * ind
  Rlow = np.fft.ifft2(Rtlow).real
  Rthigh = Rt * indr
  Rhigh = np.fft.ifft2(Rthigh).real

  Gt = np.fft.fft2(im[:,:,1])
  Gtsort = np.sort(np.abs(Gt.reshape(-1)))
  tresh = Gtsort[int(np.floor((1-keep)*len(Gtsort)))]
  ind = np.abs(Gt)>tresh
  indg = np.abs(Gt)<tresh
  Gtlow = Gt * ind
  Glow = np.fft.ifft2(Gtlow).real
  Gthigh = Gt * indg
  Ghigh = np.fft.ifft2(Gthigh).real

  Bt = np.fft.fft2(im[:,:,2])
  Btsort = np.sort(np.abs(Bt.reshape(-1)))
  tresh = Btsort[int(np.floor((1-keep)*len(Btsort)))]
  ind = np.abs(Bt)>tresh
  indb = np.abs(Bt)<tresh
  Btlow = Bt * ind
  Blow = np.fft.ifft2(Btlow).real
  Bthigh = Bt * indb
  Bhigh = np.fft.ifft2(Bthigh).real

  image = transforms.ToTensor()(im)
  image = image.unsqueeze_(0)
  image = image.to(device)
  oimage = Variable(image.clone(), requires_grad=True).to(device)

  opt = torch.optim.Adam([oimage], lr = lrlr)

  for e in range(1):
    p2p_score = model(oimage).mean()
    loss = - p2p_score
    loss.backward()
    oimage.grad[oimage.grad == None] = 0
    oimage.grad = torch.sign(oimage.grad)
    opt.step()
    oimage.data.clamp_(0., 1.)
    opt.zero_grad()

  res_image = (oimage).data.clamp_(min=0, max=1)
  res_img = (res_image.squeeze().data.cpu().numpy().transpose(1, 2, 0) * 255).astype('uint8')

  Rh = np.fft.fft2(res_img[:,:,0])
  Rh = Rh * indr
  Rhlow = np.fft.ifft2(Rh).real

  Gh = np.fft.fft2(res_img[:,:,1])
  Gh = Gh * indg
  Ghlow = np.fft.ifft2(Gh).real

  Bh = np.fft.fft2(res_img[:,:,2])
  Bh = Bh * indb
  Bhlow = np.fft.ifft2(Bh).real

  R = Rlow + Rhlow * f[:,:,0] + Rhigh * (1 - f[:,:,0])
  G = Glow + Ghlow * f[:,:,1] + Ghigh * (1 - f[:,:,1])
  B = Blow + Bhlow * f[:,:,2] + Bhigh * (1 - f[:,:,2])

  res_img = np.stack([R, G, B], axis=2)
  res_img[res_img > 255.] = 255
  res_img[res_img < 0] = 0
  return res_img.astype('uint8')

## Attack examples

In [ ]:
# additional cell before cell 11
# to use previously downloaded image to ./img/ dir
import os
if os.path.exists('Picture1.jpg'):
    shutil.move('Picture1.jpg', './img/Picture1.jpg')

In [11]:
ims = sorted(os.listdir('./img/'))

In [19]:
ref_img = cv2.imread('./img/'+ims[0])
ref_img = cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB)
adv_img = attack(ref_img, 0.1)
score_before = overall_model.predict(ref_img)['global_score']
score_after = overall_model.predict(adv_img)['global_score']

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 8))
ax[0].imshow(ref_img)
ax[1].imshow(adv_img)
ax[0].set_title('Original. PaQ-2-PiQ = ' + str(score_before))
ax[1].set_title('Attacked. PaQ-2-PiQ = ' + str(score_after))
plt.show()

In [ ]:
ref_img = cv2.imread('./img/'+ims[1])
ref_img = cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB)
adv_img = attack(ref_img, 0.1)
score_before = overall_model.predict(ref_img)['global_score']
score_after = overall_model.predict(adv_img)['global_score']

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 8))
ax[0].imshow(ref_img)
ax[1].imshow(adv_img)
ax[0].set_title('Original. PaQ-2-PiQ = ' + str(score_before))
ax[1].set_title('Attacked. PaQ-2-PiQ = ' + str(score_after))
plt.show()

In [ ]:
ref_img = cv2.imread('./img/'+ims[2])
ref_img = cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB)
adv_img = attack(ref_img, 0.1)
score_before = overall_model.predict(ref_img)['global_score']
score_after = overall_model.predict(adv_img)['global_score']

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 8))
ax[0].imshow(ref_img)
ax[1].imshow(adv_img)
ax[0].set_title('Original. PaQ-2-PiQ = ' + str(score_before))
ax[1].set_title('Attacked. PaQ-2-PiQ = ' + str(score_after))
plt.show()